# HMM and Yang–Zhang Regime Strategy

Uses an HMM and range-based volatility in a directional backtest.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
================================================================================
HMM REGIME-SWITCHING STRATEGY WITH STUDENT-T AND YANG-ZHANG VOLATILITY
================================================================================

Strategy Overview:
- Uses Hidden Markov Model with Student-t distributions to detect volatility regimes in SPY
- Executes long/short signals on QQQ based on detected regime
- Features: Log returns, intraday returns, overnight returns, Yang-Zhang volatility,
  semivariance, and Sharpe ratio with downside volatility
- Walk-forward validation with rolling 6-month train / 1-month test windows
- No look-ahead bias: All features use shift(1), signals at T-1, execution at T
- Probabilistic Sharpe Ratio (PSR) for statistical validation

Author: Trading Strategies Module
Date: October 2025
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from scipy import stats
from scipy.special import gammaln
from scipy.optimize import minimize
import warnings

warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.dates import YearLocator, MonthLocator, DateFormatter

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
try:
    from hmmlearn import hmm
    from hmmlearn.base import _BaseHMM

    HMM_AVAILABLE = True
except ImportError:
    HMM_AVAILABLE = False
    print("WARNING: hmmlearn not available. Will use custom HMM implementation.")
CONFIG = {
    "END_DATE": "2018-12-31",
    "START_DATE": "2014-01-01",
    "REGIME_ASSET": "SPY",
    "TRADING_ASSET": "QQQ",
    "BENCHMARK": "URTH",
    "ROLLING_VOL_WINDOW": 21,
    "SEMIVARIANCE_WINDOW": 21,
    "DOWNSIDE_VOL_WINDOW": 21,
    "N_STATES_MIN": 2,
    "N_STATES_MAX": 3,
    "HMM_MAX_ITER": 100,
    "HMM_TOL": 0.01,
    "DF_MIN": 2.1,
    "DF_MAX": 30.0,
    "TRAIN_MONTHS": 6,
    "TEST_MONTHS": 1,
    "MIN_TRAIN_DAYS": 60,
    "INITIAL_CAPITAL": 100000.0,
    "COMPOUNDING_MODE": False,
    "LEVERAGE": 1.0,
    "COMMISSION_PCT": 0.001,
    "MAX_POSITION_SIZE": 1.0,
    "LONG_REGIME": "low_vol",
    "SHORT_REGIME": "high_vol",
    "ROLLING_SHARPE_WINDOW": 252,
    "ROLLING_BETA_WINDOW": 126,
    "FIGURE_SIZE": (16, 10),
    "DPI": 100,
}
print("=" * 80)
print("HMM REGIME-SWITCHING STRATEGY WITH STUDENT-T AND YANG-ZHANG VOLATILITY")
print("=" * 80)
print(f"\nConfiguration:")
print(f"  Period: {CONFIG['START_DATE']} to {CONFIG['END_DATE']}")
print(f"  Regime Asset: {CONFIG['REGIME_ASSET']}")
print(f"  Trading Asset: {CONFIG['TRADING_ASSET']}")
print(f"  Benchmark: {CONFIG['BENCHMARK']}")
print(f"  Initial Capital: ${CONFIG['INITIAL_CAPITAL']:,.0f}")
print(f"  Compounding Mode: {CONFIG['COMPOUNDING_MODE']}")
print(f"  Leverage: {CONFIG['LEVERAGE']}x")
print(f"  Commission: {CONFIG['COMMISSION_PCT'] * 100:.2f}%")
print(f"  Walk-Forward: {CONFIG['TRAIN_MONTHS']}m train / {CONFIG['TEST_MONTHS']}m test")


def fetch_yahoo_data(symbol, start_date, end_date):
    """
    Fetch OHLCV data from Yahoo Finance

    Parameters:
    -----------
    symbol : str - Ticker symbol
    start_date : str - Start date in YYYY-MM-DD format
    end_date : str - End date in YYYY-MM-DD format

    Returns:
    --------
    pd.DataFrame - OHLCV data with datetime index

    Note: Yahoo Finance provides data with Open, High, Low, Close, Volume
    Critical for Yang-Zhang volatility calculation which requires OHLC

    API Endpoint: /api/v3/historical-price-full/{symbol}
    No look-ahead: Data is as-of close of each trading day
    """
    try:
        response_prices = {"historical": download_price_records(symbol, start_date, end_date)}
        data = response_prices
        if "historical" not in data or len(data["historical"]) == 0:
            print(f"Warning: No data found for {symbol}")
            return None
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date")
        df.set_index("date", inplace=True)
        df = df.rename(
            columns={
                "open": "Open",
                "high": "High",
                "low": "Low",
                "close": "Close",
                "volume": "Volume",
            }
        )
        df = df[["Open", "High", "Low", "Close", "Volume"]]
        if df.isnull().sum().sum() > 0:
            print(f"Warning: {symbol} has {df.isnull().sum().sum()} missing values")
            df = df.dropna()
        return df
    except Exception as e:
        print(f"Error fetching data for {symbol}: {str(e)}")
        return None


print("\n" + "=" * 80)
print("DOWNLOADING DATA FROM Yahoo Finance")
print("=" * 80)
extended_start = (pd.to_datetime(CONFIG["START_DATE"]) - timedelta(days=180)).strftime("%Y-%m-%d")
print(f"\nFetching {CONFIG['REGIME_ASSET']} (regime detection)...")
spy_data = fetch_yahoo_data(CONFIG["REGIME_ASSET"], extended_start, CONFIG["END_DATE"])
print(f"Fetching {CONFIG['TRADING_ASSET']} (trading execution)...")
qqq_data = fetch_yahoo_data(CONFIG["TRADING_ASSET"], extended_start, CONFIG["END_DATE"])
print(f"Fetching {CONFIG['BENCHMARK']} (benchmark comparison)...")
benchmark_data = fetch_yahoo_data(CONFIG["BENCHMARK"], extended_start, CONFIG["END_DATE"])
if spy_data is None or qqq_data is None or benchmark_data is None:
    raise ValueError("Failed to download required data. Check API key and symbols.")
print(
    f"\n{CONFIG['REGIME_ASSET']} data: {len(spy_data)} rows from {spy_data.index[0].date()} to {spy_data.index[-1].date()}"
)
print(
    f"{CONFIG['TRADING_ASSET']} data: {len(qqq_data)} rows from {qqq_data.index[0].date()} to {qqq_data.index[-1].date()}"
)
print(
    f"{CONFIG['BENCHMARK']} data: {len(benchmark_data)} rows from {benchmark_data.index[0].date()} to {benchmark_data.index[-1].date()}"
)


def calculate_yang_zhang_volatility(df, window=21):
    """
    Calculate Yang-Zhang volatility estimator

    The Yang-Zhang estimator is an extension of the Parkinson and Garman-Klass
    estimators that handles both opening jumps and drift. It has minimum estimation
    error and is independent of drift and opening gaps.

    Formula:
    σ²_YZ = σ²_o + k·σ²_c + (1-k)·σ²_rs

    Where:
    - σ²_o: Variance of opening jump (overnight returns)
    - σ²_c: Close-to-close variance
    - σ²_rs: Rogers-Satchell variance (intraday without drift)
    - k: Weighting factor = 0.34 / (1.34 + (n+1)/(n-1))

    Parameters:
    -----------
    df : pd.DataFrame with Open, High, Low, Close columns
    window : int, rolling window size

    Returns:
    --------
    pd.Series - Annualized Yang-Zhang volatility

    Reference: Yang & Zhang (2000) "Drift-Independent Volatility Estimation
    Based on High, Low, Open, and Close Prices"

    CRITICAL: No look-ahead bias - uses only past data
    """
    overnight = np.log(df["Open"] / df["Close"].shift(1))
    open_to_close = np.log(df["Close"] / df["Open"])
    rs = np.log(df["High"] / df["Close"]) * np.log(df["High"] / df["Open"]) + np.log(
        df["Low"] / df["Close"]
    ) * np.log(df["Low"] / df["Open"])
    var_overnight = overnight.rolling(window=window).var()
    var_open_to_close = open_to_close.rolling(window=window).var()
    var_rs = rs.rolling(window=window).mean()
    n = window
    k = 0.34 / (1.34 + (n + 1) / (n - 1))
    yz_variance = var_overnight + k * var_open_to_close + (1 - k) * var_rs
    yz_volatility = np.sqrt(yz_variance * 252)
    return yz_volatility


def calculate_semivariance(returns, window=21):
    """
    Calculate rolling semivariance (downside variance)

    Semivariance only considers returns below the mean, providing a measure
    of downside risk. More relevant for risk-averse investors than standard
    variance which treats upside and downside equally.

    Formula:
    Semivariance = (1/n) * Σ[min(r_i - μ, 0)]²

    Parameters:
    -----------
    returns : pd.Series - Return series
    window : int - Rolling window

    Returns:
    --------
    pd.Series - Rolling semivariance

    CRITICAL: No look-ahead - uses only past returns
    """

    def semi_var(x):
        mean = x.mean()
        downside = x[x < mean]
        if len(downside) == 0:
            return 0
        return (downside**2).mean()

    return returns.rolling(window=window).apply(semi_var, raw=False)


def calculate_downside_volatility(returns, window=21, mar=0):
    """
    Calculate downside volatility (semi-deviation)

    Used in Sortino ratio and downside-adjusted Sharpe ratio.
    Only considers returns below a Minimum Acceptable Return (MAR).

    Parameters:
    -----------
    returns : pd.Series - Return series
    window : int - Rolling window
    mar : float - Minimum acceptable return (default=0)

    Returns:
    --------
    pd.Series - Rolling downside volatility (annualized)

    CRITICAL: No look-ahead - uses only past returns
    """

    def downside_vol(x):
        downside = x[x < mar]
        if len(downside) == 0:
            return np.nan
        return np.sqrt((downside**2).mean())

    return returns.rolling(window=window).apply(downside_vol, raw=False) * np.sqrt(252)


def create_features(spy_data, window_vol=21, window_semi=21, window_sharpe=21):
    """
    Create all features for regime detection from SPY data

    ALL FEATURES MUST USE shift(1) TO PREVENT LOOK-AHEAD BIAS

    Features generated:
    1. LogReturns: log(Close_t / Close_t-1)
    2. IntradayReturns: log(Close_t / Open_t)
    3. OvernightReturns: log(Open_t / Close_t-1)
    4. YangZhangVol: Rolling Yang-Zhang volatility
    5. Semivariance: Rolling downside variance
    6. SharpeRatio_Downside: Rolling Sharpe with downside volatility

    Parameters:
    -----------
    spy_data : pd.DataFrame - SPY OHLCV data
    window_vol : int - Window for Yang-Zhang volatility
    window_semi : int - Window for semivariance
    window_sharpe : int - Window for Sharpe calculation

    Returns:
    --------
    pd.DataFrame - Features dataframe with NO look-ahead bias
    """
    features = pd.DataFrame(index=spy_data.index)
    features["LogReturns"] = np.log(spy_data["Close"] / spy_data["Close"].shift(1))
    features["IntradayReturns"] = np.log(spy_data["Close"] / spy_data["Open"])
    features["OvernightReturns"] = np.log(spy_data["Open"] / spy_data["Close"].shift(1))
    features["YangZhangVol"] = calculate_yang_zhang_volatility(spy_data, window=window_vol)
    features["Semivariance"] = calculate_semivariance(features["LogReturns"], window=window_semi)
    downside_vol = calculate_downside_volatility(features["LogReturns"], window=window_sharpe)
    rolling_mean = features["LogReturns"].rolling(window=window_sharpe).mean() * 252
    features["SharpeRatio_Downside"] = rolling_mean / downside_vol
    for col in features.columns:
        features[col] = features[col].shift(1)
    features = features.dropna()
    return features


print("\n" + "=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)
spy_features = create_features(
    spy_data,
    window_vol=CONFIG["ROLLING_VOL_WINDOW"],
    window_semi=CONFIG["SEMIVARIANCE_WINDOW"],
    window_sharpe=CONFIG["DOWNSIDE_VOL_WINDOW"],
)
print(f"\nFeatures created: {list(spy_features.columns)}")
print(f"Feature matrix shape: {spy_features.shape}")
print(f"Date range: {spy_features.index[0].date()} to {spy_features.index[-1].date()}")
print(f"\nFeature statistics:\n{spy_features.describe()}")


class StudentTHMM:
    """
    Hidden Markov Model with Student-t emission distributions

    The Student-t distribution is more robust to outliers than Gaussian,
    making it ideal for financial returns which exhibit fat tails and
    occasional extreme events.

    Model:
    - Hidden states: S_t ∈ {1, ..., K} (volatility regimes)
    - Observations: X_t (feature vector at time t)
    - Emission: X_t | S_t=k ~ StudentT(μ_k, Σ_k, ν_k)

    Parameters:
    -----------
    n_states : int - Number of hidden states (regimes)
    n_features : int - Number of features in observation vector
    max_iter : int - Maximum EM iterations
    tol : float - Convergence tolerance
    df_min : float - Minimum degrees of freedom for Student-t
    df_max : float - Maximum degrees of freedom for Student-t

    Attributes:
    -----------
    startprob_ : array (n_states,) - Initial state probabilities
    transmat_ : array (n_states, n_states) - Transition matrix
    means_ : array (n_states, n_features) - Mean vectors per state
    covars_ : array (n_states, n_features, n_features) - Covariance matrices
    dfs_ : array (n_states,) - Degrees of freedom per state
    """

    def __init__(
        self,
        n_states=2,
        n_features=1,
        max_iter=100,
        tol=0.01,
        df_min=2.1,
        df_max=30.0,
        random_state=None,
    ):
        self.n_states = n_states
        self.n_features = n_features
        self.max_iter = max_iter
        self.tol = tol
        self.df_min = df_min
        self.df_max = df_max
        self.random_state = random_state
        self.startprob_ = None
        self.transmat_ = None
        self.means_ = None
        self.covars_ = None
        self.dfs_ = None
        if random_state is not None:
            np.random.seed(random_state)

    def _initialize_parameters(self, X):
        """
        Initialize HMM parameters using k-means-style clustering

        IMPROVEMENT: Better initialization leads to faster convergence
        and reduces risk of local optima in EM algorithm
        """
        n_samples = X.shape[0]
        self.startprob_ = np.ones(self.n_states) / self.n_states
        self.transmat_ = np.ones((self.n_states, self.n_states)) * 0.1
        np.fill_diagonal(self.transmat_, 0.7)
        self.transmat_ = self.transmat_ / self.transmat_.sum(axis=1, keepdims=True)
        quantiles = np.linspace(0, 1, self.n_states + 1)[1:-1]
        init_points = np.quantile(X[:, 0], quantiles)
        distances = np.abs(X[:, 0][:, np.newaxis] - init_points)
        assignments = np.argmin(distances, axis=1)
        self.means_ = np.zeros((self.n_states, self.n_features))
        self.covars_ = np.zeros((self.n_states, self.n_features, self.n_features))
        for k in range(self.n_states):
            mask = assignments == k
            if mask.sum() > 0:
                self.means_[k] = X[mask].mean(axis=0)
                cov = np.cov(X[mask].T)
                self.covars_[k] = cov + np.eye(self.n_features) * 1e-06
            else:
                self.means_[k] = X.mean(axis=0)
                self.covars_[k] = np.cov(X.T) + np.eye(self.n_features) * 1e-06
        self.dfs_ = np.ones(self.n_states) * 5.0

    def _student_t_logpdf(self, X, mean, cov, df):
        """
        Calculate log probability density of multivariate Student-t

        Formula:
        log p(x|μ,Σ,ν) = log Γ((ν+d)/2) - log Γ(ν/2) - d/2 log(νπ)
                          - 1/2 log|Σ| - (ν+d)/2 log(1 + δ²/ν)

        Where δ² = (x-μ)ᵀ Σ⁻¹ (x-μ) is Mahalanobis distance

        Parameters:
        -----------
        X : array (n_samples, n_features)
        mean : array (n_features,)
        cov : array (n_features, n_features)
        df : float - degrees of freedom

        Returns:
        --------
        array (n_samples,) - log probabilities
        """
        n_samples, n_features = X.shape
        X_centered = X - mean
        try:
            cov_inv = np.linalg.inv(cov)
            maha_dist = np.sum(X_centered @ cov_inv * X_centered, axis=1)
        except np.linalg.LinAlgError:
            cov_inv = np.linalg.pinv(cov)
            maha_dist = np.sum(X_centered @ cov_inv * X_centered, axis=1)
        sign, logdet = np.linalg.slogdet(cov)
        if sign <= 0:
            logdet = 0
        log_prob = (
            gammaln((df + n_features) / 2)
            - gammaln(df / 2)
            - n_features / 2 * np.log(df * np.pi)
            - 0.5 * logdet
            - (df + n_features) / 2 * np.log(1 + maha_dist / df)
        )
        return log_prob

    def _compute_emission_logprob(self, X):
        """
        Compute log emission probabilities for all states

        Returns:
        --------
        array (n_samples, n_states) - log P(X_t | S_t = k)
        """
        n_samples = X.shape[0]
        log_prob = np.zeros((n_samples, self.n_states))
        for k in range(self.n_states):
            log_prob[:, k] = self._student_t_logpdf(
                X, self.means_[k], self.covars_[k], self.dfs_[k]
            )
        return log_prob

    def _forward(self, log_emission):
        """
        Forward algorithm to compute α_t(k) = P(X_1:t, S_t=k)

        Uses log-space for numerical stability
        """
        n_samples = log_emission.shape[0]
        log_alpha = np.zeros((n_samples, self.n_states))
        log_alpha[0] = np.log(self.startprob_) + log_emission[0]
        for t in range(1, n_samples):
            for k in range(self.n_states):
                log_alpha[t, k] = log_emission[t, k] + self._log_sum_exp(
                    log_alpha[t - 1] + np.log(self.transmat_[:, k])
                )
        return log_alpha

    def _backward(self, log_emission):
        """
        Backward algorithm to compute β_t(k) = P(X_t+1:T | S_t=k)

        Uses log-space for numerical stability
        """
        n_samples = log_emission.shape[0]
        log_beta = np.zeros((n_samples, self.n_states))
        for t in range(n_samples - 2, -1, -1):
            for k in range(self.n_states):
                log_beta[t, k] = self._log_sum_exp(
                    np.log(self.transmat_[k, :]) + log_emission[t + 1] + log_beta[t + 1]
                )
        return log_beta

    def _log_sum_exp(self, log_probs):
        """
        Numerically stable log-sum-exp

        log(Σ exp(x_i)) = log(Σ exp(x_i - x_max) * exp(x_max))
                        = x_max + log(Σ exp(x_i - x_max))
        """
        max_log_prob = np.max(log_probs)
        if not np.isfinite(max_log_prob):
            return -np.inf
        return max_log_prob + np.log(np.sum(np.exp(log_probs - max_log_prob)))

    def _compute_posteriors(self, log_alpha, log_beta):
        """
        Compute state posteriors γ_t(k) = P(S_t=k | X_1:T)

        γ_t(k) = α_t(k) * β_t(k) / P(X_1:T)
        """
        log_gamma = log_alpha + log_beta
        log_norm = self._log_sum_exp(log_gamma[0])
        log_gamma = log_gamma - log_norm
        return np.exp(log_gamma)

    def _compute_xi(self, log_alpha, log_beta, log_emission):
        """
        Compute ξ_t(i,j) = P(S_t=i, S_t+1=j | X_1:T)

        Used for updating transition matrix
        """
        n_samples = log_alpha.shape[0]
        log_xi = np.zeros((n_samples - 1, self.n_states, self.n_states))
        for t in range(n_samples - 1):
            for i in range(self.n_states):
                for j in range(self.n_states):
                    log_xi[t, i, j] = (
                        log_alpha[t, i]
                        + np.log(self.transmat_[i, j])
                        + log_emission[t + 1, j]
                        + log_beta[t + 1, j]
                    )
            log_norm = self._log_sum_exp(log_xi[t].flatten())
            log_xi[t] -= log_norm
        return np.exp(log_xi)

    def _m_step(self, X, gamma, xi):
        """
        M-step: Update parameters given posteriors

        Updates:
        - Start probabilities: π_k = γ_1(k)
        - Transition matrix: a_ij = Σ_t ξ_t(i,j) / Σ_t γ_t(i)
        - Means: μ_k = Σ_t γ_t(k) x_t / Σ_t γ_t(k)
        - Covariances: Σ_k = Σ_t γ_t(k) (x_t - μ_k)(x_t - μ_k)ᵀ / Σ_t γ_t(k)
        - Degrees of freedom: Optimize via MLE (iterative)
        """
        n_samples = X.shape[0]
        self.startprob_ = gamma[0] / gamma[0].sum()
        for i in range(self.n_states):
            denom = gamma[:-1, i].sum()
            if denom > 0:
                for j in range(self.n_states):
                    self.transmat_[i, j] = xi[:, i, j].sum() / denom
        self.transmat_ = self.transmat_ / self.transmat_.sum(axis=1, keepdims=True)
        for k in range(self.n_states):
            gamma_sum = gamma[:, k].sum()
            if gamma_sum > 0:
                self.means_[k] = (gamma[:, k][:, np.newaxis] * X).sum(axis=0) / gamma_sum
                X_centered = X - self.means_[k]
                self.covars_[k] = (
                    gamma[:, k][:, np.newaxis, np.newaxis]
                    * (X_centered[:, :, np.newaxis] @ X_centered[:, np.newaxis, :])
                ).sum(axis=0) / gamma_sum
                self.covars_[k] += np.eye(self.n_features) * 1e-06
                maha_dist = np.sum(X_centered @ np.linalg.inv(self.covars_[k]) * X_centered, axis=1)
                weighted_maha = (gamma[:, k] * maha_dist).sum() / gamma_sum
                self.dfs_[k] = np.clip(
                    self.n_features / weighted_maha * 10, self.df_min, self.df_max
                )

    def fit(self, X):
        """
        Fit HMM using Baum-Welch (EM) algorithm

        E-step: Compute posteriors given current parameters
        M-step: Update parameters given posteriors

        Parameters:
        -----------
        X : array (n_samples, n_features)

        Returns:
        --------
        self
        """
        self._initialize_parameters(X)
        prev_log_likelihood = -np.inf
        for iteration in range(self.max_iter):
            log_emission = self._compute_emission_logprob(X)
            log_alpha = self._forward(log_emission)
            log_beta = self._backward(log_emission)
            log_likelihood = self._log_sum_exp(log_alpha[-1])
            if np.abs(log_likelihood - prev_log_likelihood) < self.tol:
                break
            prev_log_likelihood = log_likelihood
            gamma = self._compute_posteriors(log_alpha, log_beta)
            xi = self._compute_xi(log_alpha, log_beta, log_emission)
            self._m_step(X, gamma, xi)
        return self

    def predict(self, X):
        """
        Predict most likely state sequence using Viterbi algorithm

        Viterbi finds: argmax_S P(S_1:T | X_1:T)

        Parameters:
        -----------
        X : array (n_samples, n_features)

        Returns:
        --------
        array (n_samples,) - Most likely state sequence
        """
        n_samples = X.shape[0]
        log_emission = self._compute_emission_logprob(X)
        log_delta = np.zeros((n_samples, self.n_states))
        psi = np.zeros((n_samples, self.n_states), dtype=int)
        log_delta[0] = np.log(self.startprob_) + log_emission[0]
        for t in range(1, n_samples):
            for k in range(self.n_states):
                candidates = log_delta[t - 1] + np.log(self.transmat_[:, k])
                psi[t, k] = np.argmax(candidates)
                log_delta[t, k] = np.max(candidates) + log_emission[t, k]
        states = np.zeros(n_samples, dtype=int)
        states[-1] = np.argmax(log_delta[-1])
        for t in range(n_samples - 2, -1, -1):
            states[t] = psi[t + 1, states[t + 1]]
        return states

    def score(self, X):
        """
        Compute log-likelihood of observations

        log P(X_1:T) = log Σ_k α_T(k)
        """
        log_emission = self._compute_emission_logprob(X)
        log_alpha = self._forward(log_emission)
        return self._log_sum_exp(log_alpha[-1])


def create_walk_forward_windows(start_date, end_date, train_months=6, test_months=1):
    """
    Create expanding walk-forward windows

    Structure: Train on past N months, test on next M months
    Window expands: each iteration adds more training data

    This mimics real-world scenario where we accumulate more data over time
    and reoptimize periodically.

    Parameters:
    -----------
    start_date : pd.Timestamp
    end_date : pd.Timestamp
    train_months : int
    test_months : int

    Returns:
    --------
    list of tuples (train_start, train_end, test_start, test_end)

    CRITICAL: No overlap between train and test within each window
    """
    windows = []
    train_start = start_date
    train_end = start_date + pd.DateOffset(months=train_months)
    test_start = train_end + pd.Timedelta(days=1)
    test_end = test_start + pd.DateOffset(months=test_months)
    while test_end <= end_date:
        windows.append((train_start, train_end, test_start, test_end))
        test_start = test_end + pd.Timedelta(days=1)
        test_end = test_start + pd.DateOffset(months=test_months)
        train_end = test_start - pd.Timedelta(days=1)
    return windows


def optimize_hmm_parameters(X_train):
    """
    Optimize number of HMM states on training data

    Uses BIC (Bayesian Information Criterion) to select optimal number of states:
    BIC = -2·log(L) + k·log(n)

    Where:
    - L: Likelihood
    - k: Number of parameters
    - n: Number of observations

    Lower BIC is better (balances fit quality with model complexity)

    Parameters:
    -----------
    X_train : array - Training feature matrix

    Returns:
    --------
    int - Optimal number of states
    dict - HMM parameters for best model

    CRITICAL: Only uses training data, never test data
    """
    best_bic = np.inf
    best_n_states = CONFIG["N_STATES_MIN"]
    best_model = None
    n_samples, n_features = X_train.shape
    for n_states in range(CONFIG["N_STATES_MIN"], CONFIG["N_STATES_MAX"] + 1):
        try:
            model = StudentTHMM(
                n_states=n_states,
                n_features=n_features,
                max_iter=CONFIG["HMM_MAX_ITER"],
                tol=CONFIG["HMM_TOL"],
                df_min=CONFIG["DF_MIN"],
                df_max=CONFIG["DF_MAX"],
            )
            model.fit(X_train)
            log_likelihood = model.score(X_train)
            n_params = (
                n_states
                - 1
                + n_states * (n_states - 1)
                + n_states * n_features
                + n_states * n_features * (n_features + 1) / 2
                + n_states
            )
            bic = -2 * log_likelihood + n_params * np.log(n_samples)
            if bic < best_bic:
                best_bic = bic
                best_n_states = n_states
                best_model = model
        except Exception as e:
            print(f"Warning: Failed to fit {n_states} states: {str(e)}")
            continue
    return (best_n_states, best_model)


def identify_regime_characteristics(model, X_train, states_train):
    """
    Identify which regime corresponds to high/low volatility

    Uses the mean Yang-Zhang volatility in each regime to classify them.

    Parameters:
    -----------
    model : StudentTHMM - Fitted HMM model
    X_train : array - Training features
    states_train : array - Predicted states

    Returns:
    --------
    dict - Mapping {state_id: 'low_vol' or 'high_vol'}

    ASSUMPTION: Feature 0 (YangZhangVol index) is volatility proxy
    """
    regime_chars = {}
    vol_per_state = {}
    for state in range(model.n_states):
        mask = states_train == state
        if mask.sum() > 0:
            vol_per_state[state] = X_train[mask, 3].mean()
    sorted_states = sorted(vol_per_state.items(), key=lambda x: x[1])
    if model.n_states == 2:
        regime_chars[sorted_states[0][0]] = "low_vol"
        regime_chars[sorted_states[1][0]] = "high_vol"
    elif model.n_states == 3:
        regime_chars[sorted_states[0][0]] = "low_vol"
        regime_chars[sorted_states[1][0]] = "medium_vol"
        regime_chars[sorted_states[2][0]] = "high_vol"
    return regime_chars


def generate_signals(states, regime_chars):
    """
    Generate trading signals based on regime

    Strategy Logic:
    - Low volatility regime: Long (expect mean reversion or trend continuation)
    - High volatility regime: Short or cash (defensive positioning)

    Parameters:
    -----------
    states : array - Predicted regime states
    regime_chars : dict - State to regime type mapping

    Returns:
    --------
    array - Signals {-1: short, 0: cash, 1: long}

    CRITICAL: Signals generated at T-1 for execution at T (open)
    """
    signals = np.zeros(len(states))
    for i, state in enumerate(states):
        regime_type = regime_chars[state]
        if regime_type == "low_vol":
            signals[i] = 1.0
        elif regime_type == "high_vol":
            signals[i] = -1.0
        else:
            signals[i] = 0.0
    return signals


def backtest_strategy(qqq_data, signals, signal_dates, config):
    """
    Backtest the strategy with realistic execution and costs

    EXECUTION FLOW (NO LOOK-AHEAD):
    Day T-1 (Close):
    - Portfolio valued at Close[T-1]
    - Features from T-2 data
    - Regime detected
    - Signals generated
    - Sizing info stored (uses Close[T-1] value)

    Day T (Open):
    - Execute pending signals at Open[T]
    - Use Close[T-1] sizing (no look-ahead)
    - Portfolio positions updated

    Day T (Close):
    - Portfolio valued at Close[T]
    - Returns: Close[T]/Close[T-1] - 1
    - This value becomes Close[T-1] for T+1 sizing

    Parameters:
    -----------
    qqq_data : pd.DataFrame - QQQ OHLCV data
    signals : array - Signal series
    signal_dates : DatetimeIndex - Dates when signals were generated
    config : dict - Configuration parameters

    Returns:
    --------
    pd.DataFrame - Backtest results with daily returns and metrics
    """
    backtest_df = pd.DataFrame(index=signal_dates)
    backtest_df["Signal"] = signals
    backtest_df = backtest_df.join(qqq_data[["Open", "Close"]], how="left")
    backtest_df = backtest_df.dropna()
    backtest_df["Close_prev"] = backtest_df["Close"].shift(1)
    backtest_df = backtest_df.dropna()
    backtest_df["Position"] = 0.0
    backtest_df["Capital"] = config["INITIAL_CAPITAL"]
    backtest_df["Cash"] = config["INITIAL_CAPITAL"]
    backtest_df["Holdings"] = 0.0
    backtest_df["Trade"] = 0.0
    backtest_df["Commission"] = 0.0
    backtest_df["Returns"] = 0.0
    current_capital = config["INITIAL_CAPITAL"]
    current_cash = config["INITIAL_CAPITAL"]
    current_holdings = 0.0
    current_position = 0.0
    for i in range(len(backtest_df)):
        date = backtest_df.index[i]
        signal = backtest_df["Signal"].iloc[i]
        open_price = backtest_df["Open"].iloc[i]
        close_price = backtest_df["Close"].iloc[i]
        if config["COMPOUNDING_MODE"]:
            sizing_base = current_capital
        else:
            sizing_base = config["INITIAL_CAPITAL"]
        target_position_value = (
            signal * config["LEVERAGE"] * sizing_base * config["MAX_POSITION_SIZE"]
        )
        target_shares = target_position_value / open_price if open_price > 0 else 0
        trade_shares = target_shares - current_holdings
        trade_value = trade_shares * open_price
        commission = abs(trade_value) * config["COMMISSION_PCT"]
        current_cash = current_cash - trade_value - commission
        current_holdings = target_shares
        current_position = signal
        holdings_value = current_holdings * close_price
        current_capital = current_cash + holdings_value
        if i > 0:
            prev_capital = backtest_df["Capital"].iloc[i - 1]
            returns = (current_capital - prev_capital) / prev_capital if prev_capital > 0 else 0
        else:
            returns = 0
        backtest_df.loc[date, "Position"] = current_position
        backtest_df.loc[date, "Capital"] = current_capital
        backtest_df.loc[date, "Cash"] = current_cash
        backtest_df.loc[date, "Holdings"] = current_holdings
        backtest_df.loc[date, "Trade"] = trade_shares
        backtest_df.loc[date, "Commission"] = commission
        backtest_df.loc[date, "Returns"] = returns
    return backtest_df


def calculate_probabilistic_sharpe_ratio(returns, benchmark_sr=0.0):
    """
    Calculate Probabilistic Sharpe Ratio (PSR)

    PSR estimates the probability that the observed Sharpe ratio exceeds
    a benchmark Sharpe ratio, accounting for estimation uncertainty.

    Formula (Bailey & López de Prado, 2012):
    PSR = Φ(√(T-1) · (SR - SR*) / √(1 - γ₃·SR + (γ₄-1)/4·SR²))

    Where:
    - Φ: Standard normal CDF
    - T: Number of observations
    - SR: Observed Sharpe ratio
    - SR*: Benchmark Sharpe ratio
    - γ₃: Skewness of returns
    - γ₄: Kurtosis of returns

    Interpretation:
    - PSR > 0.95: Very high confidence that strategy has skill
    - PSR > 0.90: High confidence
    - PSR < 0.50: Low confidence (likely due to luck)

    Parameters:
    -----------
    returns : pd.Series - Return series
    benchmark_sr : float - Benchmark Sharpe to compare against (default 0)

    Returns:
    --------
    float - Probabilistic Sharpe Ratio [0, 1]

    Reference: Bailey, D.H. and López de Prado, M. (2012)
    "The Sharpe Ratio Efficient Frontier"
    """
    returns = returns.dropna()
    if len(returns) < 2:
        return 0.0
    T = len(returns)
    mean_ret = returns.mean()
    std_ret = returns.std()
    sr_observed = mean_ret * 252 / (std_ret * np.sqrt(252)) if std_ret > 0 else 0
    skew = returns.skew()
    kurt = returns.kurtosis()
    numerator = np.sqrt(T - 1) * (sr_observed - benchmark_sr)
    denominator_term = 1 - skew * sr_observed + kurt / 4 * sr_observed**2
    if denominator_term <= 0:
        return 0.5
    denominator = np.sqrt(denominator_term)
    z_score = numerator / denominator if denominator > 0 else 0
    psr = stats.norm.cdf(z_score)
    return psr


def calculate_deflated_sharpe_ratio(returns, n_trials=1):
    """
    Calculate Deflated Sharpe Ratio (DSR)

    DSR adjusts for multiple testing bias (data mining).
    Accounts for the fact that we may have tried many strategies
    and selected the best one.

    Formula (Bailey et al., 2014):
    DSR = Φ((SR - E[max SR]) / √Var[max SR])

    Where E[max SR] and Var[max SR] depend on:
    - Number of trials (strategies tested)
    - Number of observations
    - Moments of returns

    Parameters:
    -----------
    returns : pd.Series - Return series
    n_trials : int - Number of strategies tested (default=1, no adjustment)

    Returns:
    --------
    float - Deflated Sharpe Ratio [0, 1]

    Reference: Bailey, D.H., Borwein, J., López de Prado, M., & Zhu, Q.J. (2014)
    "The Deflated Sharpe Ratio: Correcting for Selection Bias, Backtest Overfitting"
    """
    if n_trials <= 1:
        return calculate_probabilistic_sharpe_ratio(returns)
    psr = calculate_probabilistic_sharpe_ratio(returns)
    adjusted_psr = psr / np.sqrt(n_trials)
    return np.clip(adjusted_psr, 0, 1)


def calculate_performance_metrics(backtest_df, benchmark_df):
    """
    Calculate comprehensive performance metrics

    Metrics:
    - Total Return
    - Annualized Return
    - Annualized Volatility
    - Sharpe Ratio
    - Sortino Ratio
    - Maximum Drawdown
    - Calmar Ratio
    - Win Rate
    - Profit Factor
    - Probabilistic Sharpe Ratio

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy backtest results
    benchmark_df : pd.DataFrame - Benchmark data

    Returns:
    --------
    dict - Performance metrics
    """
    returns = backtest_df["Returns"]
    benchmark_returns = benchmark_df["Close"].pct_change()
    benchmark_returns = benchmark_returns.reindex(returns.index).fillna(0)
    total_return = backtest_df["Capital"].iloc[-1] / CONFIG["INITIAL_CAPITAL"] - 1
    n_years = len(returns) / 252
    annualized_return = (1 + total_return) ** (1 / n_years) - 1 if n_years > 0 else 0
    volatility = returns.std() * np.sqrt(252)
    sharpe = annualized_return / volatility if volatility > 0 else 0
    downside_returns = returns[returns < 0]
    downside_vol = (
        downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else volatility
    )
    sortino = annualized_return / downside_vol if downside_vol > 0 else 0
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    calmar = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0
    win_rate = (returns > 0).sum() / len(returns) if len(returns) > 0 else 0
    gross_profit = returns[returns > 0].sum()
    gross_loss = abs(returns[returns < 0].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf
    psr = calculate_probabilistic_sharpe_ratio(returns)
    bench_total_return = benchmark_df["Close"].iloc[-1] / benchmark_df["Close"].iloc[0] - 1
    bench_ann_return = (1 + bench_total_return) ** (1 / n_years) - 1 if n_years > 0 else 0
    bench_vol = benchmark_returns.std() * np.sqrt(252)
    bench_sharpe = bench_ann_return / bench_vol if bench_vol > 0 else 0
    metrics = {
        "Total Return": total_return,
        "Annualized Return": annualized_return,
        "Volatility": volatility,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Max Drawdown": max_drawdown,
        "Calmar Ratio": calmar,
        "Win Rate": win_rate,
        "Profit Factor": profit_factor,
        "PSR": psr,
        "Benchmark Total Return": bench_total_return,
        "Benchmark Ann Return": bench_ann_return,
        "Benchmark Sharpe": bench_sharpe,
        "Alpha": annualized_return - bench_ann_return,
    }
    return metrics


print("\n" + "=" * 80)
print("WALK-FORWARD BACKTEST")
print("=" * 80)
wf_start = pd.to_datetime(CONFIG["START_DATE"])
wf_end = pd.to_datetime(CONFIG["END_DATE"])
windows = create_walk_forward_windows(
    wf_start, wf_end, CONFIG["TRAIN_MONTHS"], CONFIG["TEST_MONTHS"]
)
print(f"\nCreated {len(windows)} walk-forward windows")
all_results = []
all_signals = []
all_dates = []
for window_idx, (train_start, train_end, test_start, test_end) in enumerate(windows):
    print(f"\nWindow {window_idx + 1}/{len(windows)}")
    print(f"  Train: {train_start.date()} to {train_end.date()}")
    print(f"  Test:  {test_start.date()} to {test_end.date()}")
    train_mask = (spy_features.index >= train_start) & (spy_features.index <= train_end)
    X_train = spy_features[train_mask].values
    if len(X_train) < CONFIG["MIN_TRAIN_DAYS"]:
        print(f"  Skipping: insufficient training data ({len(X_train)} days)")
        continue
    print(f"  Optimizing HMM parameters...")
    best_n_states, best_model = optimize_hmm_parameters(X_train)
    print(f"  Optimal number of states: {best_n_states}")
    if best_model is None:
        print(f"  Skipping: HMM optimization failed")
        continue
    states_train = best_model.predict(X_train)
    regime_chars = identify_regime_characteristics(best_model, X_train, states_train)
    print(f"  Regime characteristics: {regime_chars}")
    test_mask = (spy_features.index >= test_start) & (spy_features.index <= test_end)
    X_test = spy_features[test_mask].values
    test_dates = spy_features[test_mask].index
    if len(X_test) == 0:
        print(f"  Skipping: no test data")
        continue
    states_test = best_model.predict(X_test)
    signals_test = generate_signals(states_test, regime_chars)
    all_signals.extend(signals_test)
    all_dates.extend(test_dates)
    print(f"  Generated {len(signals_test)} signals")
    print(
        f"  Long: {(signals_test == 1).sum()}, Short: {(signals_test == -1).sum()}, Cash: {(signals_test == 0).sum()}"
    )
print(f"\n" + "=" * 80)
print("FINAL BACKTEST")
print("=" * 80)
if len(all_signals) == 0:
    raise ValueError("No signals generated. Check walk-forward windows and data.")
signals_series = np.array(all_signals)
dates_series = pd.DatetimeIndex(all_dates)
print(f"\nTotal signals: {len(signals_series)}")
print(f"Date range: {dates_series[0].date()} to {dates_series[-1].date()}")
backtest_results = backtest_strategy(qqq_data, signals_series, dates_series, CONFIG)
print(f"\nBacktest complete: {len(backtest_results)} days")
print(f"Final capital: ${backtest_results['Capital'].iloc[-1]:,.2f}")
benchmark_subset = benchmark_data.loc[backtest_results.index[0] : backtest_results.index[-1]]
metrics = calculate_performance_metrics(backtest_results, benchmark_subset)
print(f"\n" + "=" * 80)
print("PERFORMANCE METRICS")
print("=" * 80)
print(f"\nStrategy Performance:")
print(f"  Total Return:      {metrics['Total Return']:.2%}")
print(f"  Annualized Return: {metrics['Annualized Return']:.2%}")
print(f"  Volatility:        {metrics['Volatility']:.2%}")
print(f"  Sharpe Ratio:      {metrics['Sharpe Ratio']:.3f}")
print(f"  Sortino Ratio:     {metrics['Sortino Ratio']:.3f}")
print(f"  Max Drawdown:      {metrics['Max Drawdown']:.2%}")
print(f"  Calmar Ratio:      {metrics['Calmar Ratio']:.3f}")
print(f"  Win Rate:          {metrics['Win Rate']:.2%}")
print(f"  Profit Factor:     {metrics['Profit Factor']:.3f}")
print(f"  PSR (p>0):         {metrics['PSR']:.3f}")
print(f"\nBenchmark ({CONFIG['BENCHMARK']}):")
print(f"  Total Return:      {metrics['Benchmark Total Return']:.2%}")
print(f"  Annualized Return: {metrics['Benchmark Ann Return']:.2%}")
print(f"  Sharpe Ratio:      {metrics['Benchmark Sharpe']:.3f}")
print(f"\nAlpha vs Benchmark:  {metrics['Alpha']:.2%}")
output_df = backtest_results[["Signal", "Position", "Capital", "Returns", "Commission"]]
output_df.to_csv("hmm_regime_switching_backtest.csv")
print(f"\nResults saved to: hmm_regime_switching_backtest.csv")


def plot_equity_curve(backtest_df, benchmark_df, config):
    """
    Plot equity curve comparison between strategy and benchmark

    Shows cumulative performance over time with key drawdown periods highlighted

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy results
    benchmark_df : pd.DataFrame - Benchmark data
    config : dict - Configuration dictionary
    """
    fig, ax = plt.subplots(figsize=CONFIG["FIGURE_SIZE"])
    strategy_equity = backtest_df["Capital"] / CONFIG["INITIAL_CAPITAL"]
    bench_aligned = benchmark_df["Close"].reindex(backtest_df.index).ffill()
    benchmark_equity = bench_aligned / bench_aligned.iloc[0]
    ax.plot(backtest_df.index, strategy_equity, label="HMM Strategy", linewidth=2, color="#2E86AB")
    ax.plot(
        backtest_df.index,
        benchmark_equity,
        label=f"{config['BENCHMARK']} Benchmark",
        linewidth=2,
        color="#A23B72",
        alpha=0.7,
    )
    ax.set_title("Equity Curve: Strategy vs Benchmark", fontsize=16, fontweight="bold", pad=20)
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Cumulative Return (Normalized)", fontsize=12)
    ax.legend(loc="upper left", fontsize=11, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.2f}x"))
    final_strat = strategy_equity.iloc[-1]
    final_bench = benchmark_equity.iloc[-1]
    ax.annotate(
        f"Strategy: {final_strat:.2f}x",
        xy=(backtest_df.index[-1], final_strat),
        xytext=(10, 10),
        textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#2E86AB", alpha=0.7),
        color="white",
        fontweight="bold",
    )
    ax.annotate(
        f"Benchmark: {final_bench:.2f}x",
        xy=(backtest_df.index[-1], final_bench),
        xytext=(10, -20),
        textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#A23B72", alpha=0.7),
        color="white",
        fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig("equity_curve.png", dpi=CONFIG["DPI"], bbox_inches="tight")
    plt.show()
    print("\nEquity curve saved to: equity_curve.png")


def plot_drawdown(backtest_df):
    """
    Plot underwater (drawdown) chart

    Shows the depth and duration of strategy drawdowns

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy results
    """
    fig, ax = plt.subplots(figsize=CONFIG["FIGURE_SIZE"])
    cumulative = (1 + backtest_df["Returns"]).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    ax.fill_between(
        backtest_df.index, 0, drawdown * 100, color="#C1292E", alpha=0.6, label="Drawdown"
    )
    ax.plot(backtest_df.index, drawdown * 100, color="#8B0000", linewidth=1.5)
    ax.set_title("Underwater Plot (Drawdown)", fontsize=16, fontweight="bold", pad=20)
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Drawdown (%)", fontsize=12)
    ax.legend(loc="lower right", fontsize=11)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.axhline(y=0, color="black", linewidth=1)
    max_dd_idx = drawdown.idxmin()
    max_dd_val = drawdown.min() * 100
    ax.annotate(
        f"Max DD: {max_dd_val:.2f}%",
        xy=(max_dd_idx, max_dd_val),
        xytext=(20, -20),
        textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color="red", lw=2),
        bbox=dict(boxstyle="round,pad=0.5", facecolor="yellow", alpha=0.8),
        fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig("drawdown.png", dpi=CONFIG["DPI"], bbox_inches="tight")
    plt.show()
    print("Drawdown chart saved to: drawdown.png")


def plot_rolling_sharpe(backtest_df, window=252):
    """
    Plot rolling Sharpe ratio

    Shows time-varying risk-adjusted performance

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy results
    window : int - Rolling window size (default 252 = 1 year)
    """
    fig, ax = plt.subplots(figsize=CONFIG["FIGURE_SIZE"])
    returns = backtest_df["Returns"]
    rolling_mean = returns.rolling(window=window).mean() * 252
    rolling_std = returns.rolling(window=window).std() * np.sqrt(252)
    rolling_sharpe = rolling_mean / rolling_std
    ax.plot(
        backtest_df.index,
        rolling_sharpe,
        linewidth=2,
        color="#F18F01",
        label=f"{window}-Day Rolling Sharpe",
    )
    ax.axhline(y=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax.axhline(y=1, color="green", linestyle="--", linewidth=1, alpha=0.5, label="Sharpe = 1.0")
    ax.axhline(y=2, color="darkgreen", linestyle="--", linewidth=1, alpha=0.5, label="Sharpe = 2.0")
    ax.set_title(
        f"Rolling Sharpe Ratio ({window}-Day Window)", fontsize=16, fontweight="bold", pad=20
    )
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Sharpe Ratio", fontsize=12)
    ax.legend(loc="best", fontsize=11)
    ax.grid(True, alpha=0.3, linestyle="--")
    mean_sharpe = rolling_sharpe.mean()
    ax.axhline(
        y=mean_sharpe,
        color="red",
        linestyle=":",
        linewidth=2,
        label=f"Mean Sharpe: {mean_sharpe:.2f}",
    )
    ax.legend(loc="best", fontsize=11)
    plt.tight_layout()
    plt.savefig("rolling_sharpe.png", dpi=CONFIG["DPI"], bbox_inches="tight")
    plt.show()
    print("Rolling Sharpe chart saved to: rolling_sharpe.png")


def plot_rolling_beta(backtest_df, benchmark_df, window=126):
    """
    Plot rolling beta vs benchmark

    Shows time-varying market exposure/correlation

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy results
    benchmark_df : pd.DataFrame - Benchmark data
    window : int - Rolling window size (default 126 = 6 months)
    """
    fig, ax = plt.subplots(figsize=CONFIG["FIGURE_SIZE"])
    strategy_returns = backtest_df["Returns"]
    bench_returns = benchmark_df["Close"].pct_change()
    bench_returns = bench_returns.reindex(backtest_df.index).fillna(0)
    rolling_cov = strategy_returns.rolling(window=window).cov(bench_returns)
    rolling_var = bench_returns.rolling(window=window).var()
    rolling_beta = rolling_cov / rolling_var
    ax.plot(
        backtest_df.index,
        rolling_beta,
        linewidth=2,
        color="#6A4C93",
        label=f"{window}-Day Rolling Beta",
    )
    ax.axhline(y=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax.axhline(
        y=1, color="red", linestyle="--", linewidth=1, alpha=0.5, label="Beta = 1.0 (Market)"
    )
    ax.fill_between(
        backtest_df.index,
        0,
        rolling_beta,
        where=rolling_beta > 0,
        alpha=0.2,
        color="green",
        label="Positive Beta",
    )
    ax.fill_between(
        backtest_df.index,
        0,
        rolling_beta,
        where=rolling_beta < 0,
        alpha=0.2,
        color="red",
        label="Negative Beta",
    )
    ax.set_title(
        f"Rolling Beta vs {CONFIG['BENCHMARK']} ({window}-Day Window)",
        fontsize=16,
        fontweight="bold",
        pad=20,
    )
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Beta", fontsize=12)
    ax.legend(loc="best", fontsize=11)
    ax.grid(True, alpha=0.3, linestyle="--")
    mean_beta = rolling_beta.mean()
    ax.axhline(
        y=mean_beta, color="orange", linestyle=":", linewidth=2, label=f"Mean Beta: {mean_beta:.2f}"
    )
    ax.legend(loc="best", fontsize=11)
    plt.tight_layout()
    plt.savefig("rolling_beta.png", dpi=CONFIG["DPI"], bbox_inches="tight")
    plt.show()
    print("Rolling beta chart saved to: rolling_beta.png")


def plot_monthly_returns_heatmap(backtest_df):
    """
    Plot heatmap of monthly returns by year

    Shows seasonality and performance patterns across time

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy results
    """
    monthly_returns = backtest_df["Returns"].resample("ME").apply(lambda x: (1 + x).prod() - 1)
    monthly_returns_df = pd.DataFrame(
        {
            "Year": monthly_returns.index.year,
            "Month": monthly_returns.index.month,
            "Return": monthly_returns.values,
        }
    )
    pivot_table = monthly_returns_df.pivot(index="Year", columns="Month", values="Return")
    pivot_table = pivot_table * 100
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(
        pivot_table,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        center=0,
        cbar_kws={"label": "Monthly Return (%)"},
        linewidths=0.5,
        linecolor="gray",
        vmin=-10,
        vmax=10,
        ax=ax,
    )
    ax.set_title("Monthly Returns Heatmap (%)", fontsize=16, fontweight="bold", pad=20)
    ax.set_xlabel("Month", fontsize=12)
    ax.set_ylabel("Year", fontsize=12)
    month_names = [
        "Jan",
        "Feb",
        "Mar",
        "Apr",
        "May",
        "Jun",
        "Jul",
        "Aug",
        "Sep",
        "Oct",
        "Nov",
        "Dec",
    ]
    ax.set_xticklabels(month_names, rotation=0)
    plt.tight_layout()
    plt.savefig("monthly_returns_heatmap.png", dpi=CONFIG["DPI"], bbox_inches="tight")
    plt.show()
    print("Monthly returns heatmap saved to: monthly_returns_heatmap.png")


def plot_annual_returns_bar(backtest_df, benchmark_df):
    """
    Plot annual returns comparison bar chart

    Compares strategy vs benchmark returns by year

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy results
    benchmark_df : pd.DataFrame - Benchmark data
    """
    annual_returns_strat = backtest_df["Returns"].resample("YE").apply(lambda x: (1 + x).prod() - 1)
    bench_returns = benchmark_df["Close"].pct_change()
    bench_returns_aligned = bench_returns.reindex(backtest_df.index).fillna(0)
    annual_returns_bench = bench_returns_aligned.resample("YE").apply(lambda x: (1 + x).prod() - 1)
    years = annual_returns_strat.index.year
    fig, ax = plt.subplots(figsize=CONFIG["FIGURE_SIZE"])
    x = np.arange(len(years))
    width = 0.35
    bars1 = ax.bar(
        x - width / 2,
        annual_returns_strat.values * 100,
        width,
        label="HMM Strategy",
        color="#2E86AB",
        alpha=0.8,
    )
    bars2 = ax.bar(
        x + width / 2,
        annual_returns_bench.values * 100,
        width,
        label=f"{CONFIG['BENCHMARK']} Benchmark",
        color="#A23B72",
        alpha=0.8,
    )
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                height,
                f"{height:.1f}%",
                ha="center",
                va="bottom" if height > 0 else "top",
                fontsize=9,
                fontweight="bold",
            )
    ax.set_title("Annual Returns: Strategy vs Benchmark", fontsize=16, fontweight="bold", pad=20)
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Annual Return (%)", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(years, rotation=45)
    ax.legend(loc="best", fontsize=11)
    ax.grid(True, alpha=0.3, linestyle="--", axis="y")
    ax.axhline(y=0, color="black", linewidth=1)
    plt.tight_layout()
    plt.savefig("annual_returns_bar.png", dpi=CONFIG["DPI"], bbox_inches="tight")
    plt.show()
    print("Annual returns bar chart saved to: annual_returns_bar.png")


def plot_return_distribution(backtest_df, benchmark_df):
    """
    Plot return distribution comparison

    Shows histogram and density of daily returns

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy results
    benchmark_df : pd.DataFrame - Benchmark data
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    strategy_returns = backtest_df["Returns"] * 100
    bench_returns = benchmark_df["Close"].pct_change() * 100
    bench_returns_aligned = bench_returns.reindex(backtest_df.index).dropna()
    ax1.hist(
        strategy_returns.dropna(),
        bins=50,
        alpha=0.6,
        label="HMM Strategy",
        color="#2E86AB",
        edgecolor="black",
        density=True,
    )
    ax1.hist(
        bench_returns_aligned,
        bins=50,
        alpha=0.6,
        label=f"{CONFIG['BENCHMARK']} Benchmark",
        color="#A23B72",
        edgecolor="black",
        density=True,
    )
    ax1.set_title("Daily Returns Distribution", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Daily Return (%)", fontsize=12)
    ax1.set_ylabel("Density", fontsize=12)
    ax1.legend(loc="best", fontsize=11)
    ax1.grid(True, alpha=0.3, linestyle="--")
    ax1.axvline(x=0, color="black", linestyle="--", linewidth=1)
    strat_mean = strategy_returns.mean()
    strat_std = strategy_returns.std()
    bench_mean = bench_returns_aligned.mean()
    bench_std = bench_returns_aligned.std()
    stats_text = f"Strategy: μ={strat_mean:.3f}%, σ={strat_std:.3f}%\n"
    stats_text += f"Benchmark: μ={bench_mean:.3f}%, σ={bench_std:.3f}%"
    ax1.text(
        0.02,
        0.98,
        stats_text,
        transform=ax1.transAxes,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
        fontsize=10,
    )
    from scipy import stats as sp_stats

    strat_sorted = np.sort(strategy_returns.dropna())
    bench_sorted = np.sort(bench_returns_aligned)
    n_strat = len(strat_sorted)
    n_bench = len(bench_sorted)
    theoretical_strat = sp_stats.norm.ppf(np.linspace(0.01, 0.99, n_strat))
    theoretical_bench = sp_stats.norm.ppf(np.linspace(0.01, 0.99, n_bench))
    ax2.scatter(
        theoretical_strat, strat_sorted, alpha=0.5, s=10, label="HMM Strategy", color="#2E86AB"
    )
    ax2.scatter(
        theoretical_bench,
        bench_sorted,
        alpha=0.5,
        s=10,
        label=f"{CONFIG['BENCHMARK']}",
        color="#A23B72",
    )
    min_val = min(theoretical_strat.min(), theoretical_bench.min())
    max_val = max(theoretical_strat.max(), theoretical_bench.max())
    ax2.plot(
        [min_val, max_val],
        [min_val * strat_std + strat_mean, max_val * strat_std + strat_mean],
        "r--",
        linewidth=2,
        label="Normal Distribution",
    )
    ax2.set_title("Q-Q Plot vs Normal Distribution", fontsize=14, fontweight="bold")
    ax2.set_xlabel("Theoretical Quantiles", fontsize=12)
    ax2.set_ylabel("Sample Quantiles (%)", fontsize=12)
    ax2.legend(loc="best", fontsize=11)
    ax2.grid(True, alpha=0.3, linestyle="--")
    plt.tight_layout()
    plt.savefig("return_distribution.png", dpi=CONFIG["DPI"], bbox_inches="tight")
    plt.show()
    print("Return distribution charts saved to: return_distribution.png")


def plot_position_analysis(backtest_df):
    """
    Plot position analysis showing long/short/cash allocation over time

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy results
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))
    positions = backtest_df["Position"]
    colors = []
    for pos in positions:
        if pos > 0:
            colors.append("#00A86B")
        elif pos < 0:
            colors.append("#C41E3A")
        else:
            colors.append("#FFD700")
    ax1.bar(backtest_df.index, positions, color=colors, width=1, alpha=0.7, edgecolor="none")
    ax1.set_title("Position Allocation Over Time", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Date", fontsize=12)
    ax1.set_ylabel("Position", fontsize=12)
    ax1.set_ylim(-1.5, 1.5)
    ax1.set_yticks([-1, 0, 1])
    ax1.set_yticklabels(["Short", "Cash", "Long"])
    ax1.grid(True, alpha=0.3, linestyle="--", axis="x")
    ax1.axhline(y=0, color="black", linewidth=1)
    from matplotlib.patches import Patch

    legend_elements = [
        Patch(facecolor="#00A86B", label="Long"),
        Patch(facecolor="#FFD700", label="Cash"),
        Patch(facecolor="#C41E3A", label="Short"),
    ]
    ax1.legend(handles=legend_elements, loc="upper right", fontsize=11)
    position_counts = positions.value_counts()
    position_pcts = position_counts / len(positions) * 100
    labels = []
    sizes = []
    colors_pie = []
    for pos, pct in position_pcts.items():
        if pos > 0:
            labels.append(f"Long ({pct:.1f}%)")
            sizes.append(pct)
            colors_pie.append("#00A86B")
        elif pos < 0:
            labels.append(f"Short ({pct:.1f}%)")
            sizes.append(pct)
            colors_pie.append("#C41E3A")
        else:
            labels.append(f"Cash ({pct:.1f}%)")
            sizes.append(pct)
            colors_pie.append("#FFD700")
    ax2.pie(
        sizes,
        labels=labels,
        colors=colors_pie,
        autopct="%1.1f%%",
        startangle=90,
        textprops={"fontsize": 12, "fontweight": "bold"},
    )
    ax2.set_title("Position Allocation Distribution", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("position_analysis.png", dpi=CONFIG["DPI"], bbox_inches="tight")
    plt.show()
    print("Position analysis saved to: position_analysis.png")


def plot_rolling_volatility(backtest_df, benchmark_df, window=21):
    """
    Plot rolling volatility comparison

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy results
    benchmark_df : pd.DataFrame - Benchmark data
    window : int - Rolling window size (default 21 = 1 month)
    """
    fig, ax = plt.subplots(figsize=CONFIG["FIGURE_SIZE"])
    strategy_vol = backtest_df["Returns"].rolling(window=window).std() * np.sqrt(252) * 100
    bench_returns = benchmark_df["Close"].pct_change()
    bench_returns_aligned = bench_returns.reindex(backtest_df.index).fillna(0)
    bench_vol = bench_returns_aligned.rolling(window=window).std() * np.sqrt(252) * 100
    ax.plot(
        backtest_df.index,
        strategy_vol,
        linewidth=2,
        color="#2E86AB",
        label="HMM Strategy",
        alpha=0.8,
    )
    ax.plot(
        backtest_df.index,
        bench_vol,
        linewidth=2,
        color="#A23B72",
        label=f"{CONFIG['BENCHMARK']} Benchmark",
        alpha=0.8,
    )
    ax.fill_between(
        backtest_df.index,
        strategy_vol,
        bench_vol,
        where=strategy_vol < bench_vol,
        alpha=0.3,
        color="green",
        interpolate=True,
        label="Lower Vol",
    )
    ax.fill_between(
        backtest_df.index,
        strategy_vol,
        bench_vol,
        where=strategy_vol >= bench_vol,
        alpha=0.3,
        color="red",
        interpolate=True,
        label="Higher Vol",
    )
    ax.set_title(
        f"Rolling Volatility Comparison ({window}-Day Window)",
        fontsize=16,
        fontweight="bold",
        pad=20,
    )
    ax.set_xlabel("Date", fontsize=12)
    ax.set_ylabel("Annualized Volatility (%)", fontsize=12)
    ax.legend(loc="best", fontsize=11)
    ax.grid(True, alpha=0.3, linestyle="--")
    plt.tight_layout()
    plt.savefig("rolling_volatility.png", dpi=CONFIG["DPI"], bbox_inches="tight")
    plt.show()
    print("Rolling volatility chart saved to: rolling_volatility.png")


def create_comprehensive_dashboard(backtest_df, benchmark_df, metrics, config):
    """
    Create comprehensive dashboard with multiple subplots

    All-in-one visualization showing key performance metrics

    Parameters:
    -----------
    backtest_df : pd.DataFrame - Strategy results
    benchmark_df : pd.DataFrame - Benchmark data
    metrics : dict - Performance metrics
    config : dict - Configuration dictionary
    """
    fig = plt.figure(figsize=(20, 12))
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, :])
    strategy_equity = backtest_df["Capital"] / config["INITIAL_CAPITAL"]
    bench_aligned = benchmark_df["Close"].reindex(backtest_df.index).ffill()
    benchmark_equity = bench_aligned / bench_aligned.iloc[0]
    ax1.plot(backtest_df.index, strategy_equity, label="HMM Strategy", linewidth=2, color="#2E86AB")
    ax1.plot(
        backtest_df.index,
        benchmark_equity,
        label=f"{config['BENCHMARK']}",
        linewidth=2,
        color="#A23B72",
        alpha=0.7,
    )
    ax1.set_title("Equity Curve", fontsize=14, fontweight="bold")
    ax1.legend(loc="upper left", fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylabel("Cumulative Return", fontsize=11)
    ax2 = fig.add_subplot(gs[1, 0])
    cumulative = (1 + backtest_df["Returns"]).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max * 100
    ax2.fill_between(backtest_df.index, 0, drawdown, color="#C1292E", alpha=0.6)
    ax2.set_title("Drawdown", fontsize=12, fontweight="bold")
    ax2.set_ylabel("Drawdown (%)", fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax3 = fig.add_subplot(gs[1, 1])
    returns = backtest_df["Returns"]
    rolling_mean = returns.rolling(window=252).mean() * 252
    rolling_std = returns.rolling(window=252).std() * np.sqrt(252)
    rolling_sharpe = rolling_mean / rolling_std
    ax3.plot(backtest_df.index, rolling_sharpe, linewidth=2, color="#F18F01")
    ax3.axhline(y=0, color="black", linestyle="--", alpha=0.5)
    ax3.axhline(y=1, color="green", linestyle="--", alpha=0.5)
    ax3.set_title("Rolling Sharpe (1Y)", fontsize=12, fontweight="bold")
    ax3.set_ylabel("Sharpe Ratio", fontsize=10)
    ax3.grid(True, alpha=0.3)
    ax4 = fig.add_subplot(gs[1, 2])
    bench_returns = benchmark_df["Close"].pct_change()
    bench_returns_aligned = bench_returns.reindex(backtest_df.index).fillna(0)
    rolling_cov = returns.rolling(window=126).cov(bench_returns_aligned)
    rolling_var = bench_returns_aligned.rolling(window=126).var()
    rolling_beta = rolling_cov / rolling_var
    ax4.plot(backtest_df.index, rolling_beta, linewidth=2, color="#6A4C93")
    ax4.axhline(y=1, color="red", linestyle="--", alpha=0.5)
    ax4.axhline(y=0, color="black", linestyle="--", alpha=0.5)
    ax4.set_title("Rolling Beta (6M)", fontsize=12, fontweight="bold")
    ax4.set_ylabel("Beta", fontsize=10)
    ax4.grid(True, alpha=0.3)
    ax5 = fig.add_subplot(gs[2, 0])
    strategy_returns_pct = returns * 100
    ax5.hist(strategy_returns_pct.dropna(), bins=50, alpha=0.7, color="#2E86AB", edgecolor="black")
    ax5.axvline(x=0, color="red", linestyle="--", linewidth=2)
    ax5.set_title("Return Distribution", fontsize=12, fontweight="bold")
    ax5.set_xlabel("Daily Return (%)", fontsize=10)
    ax5.set_ylabel("Frequency", fontsize=10)
    ax5.grid(True, alpha=0.3)
    ax6 = fig.add_subplot(gs[2, 1])
    position_counts = backtest_df["Position"].value_counts()
    position_pcts = position_counts / len(backtest_df["Position"]) * 100
    labels = []
    sizes = []
    colors_pie = []
    for pos, pct in position_pcts.items():
        if pos > 0:
            labels.append(f"Long")
            sizes.append(pct)
            colors_pie.append("#00A86B")
        elif pos < 0:
            labels.append(f"Short")
            sizes.append(pct)
            colors_pie.append("#C41E3A")
        else:
            labels.append(f"Cash")
            sizes.append(pct)
            colors_pie.append("#FFD700")
    ax6.pie(
        sizes,
        labels=labels,
        colors=colors_pie,
        autopct="%1.1f%%",
        startangle=90,
        textprops={"fontsize": 10},
    )
    ax6.set_title("Position Allocation", fontsize=12, fontweight="bold")
    ax7 = fig.add_subplot(gs[2, 2])
    ax7.axis("off")
    metrics_data = [
        ["Metric", "Strategy", "Benchmark"],
        [
            "Total Return",
            f"{metrics['Total Return']:.2%}",
            f"{metrics['Benchmark Total Return']:.2%}",
        ],
        [
            "Ann. Return",
            f"{metrics['Annualized Return']:.2%}",
            f"{metrics['Benchmark Ann Return']:.2%}",
        ],
        ["Volatility", f"{metrics['Volatility']:.2%}", "-"],
        ["Sharpe Ratio", f"{metrics['Sharpe Ratio']:.2f}", f"{metrics['Benchmark Sharpe']:.2f}"],
        ["Sortino Ratio", f"{metrics['Sortino Ratio']:.2f}", "-"],
        ["Max Drawdown", f"{metrics['Max Drawdown']:.2%}", "-"],
        ["Calmar Ratio", f"{metrics['Calmar Ratio']:.2f}", "-"],
        ["Win Rate", f"{metrics['Win Rate']:.2%}", "-"],
        ["PSR", f"{metrics['PSR']:.2f}", "-"],
        ["Alpha", f"{metrics['Alpha']:.2%}", "-"],
    ]
    table = ax7.table(
        cellText=metrics_data, cellLoc="center", loc="center", colWidths=[0.4, 0.3, 0.3]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    for i in range(3):
        table[0, i].set_facecolor("#4472C4")
        table[0, i].set_text_props(weight="bold", color="white")
    for i in range(1, len(metrics_data)):
        for j in range(3):
            if i % 2 == 0:
                table[i, j].set_facecolor("#E7E6E6")
    ax7.set_title("Performance Metrics", fontsize=12, fontweight="bold", pad=20)
    fig.suptitle(
        "HMM REGIME-SWITCHING STRATEGY - COMPREHENSIVE DASHBOARD",
        fontsize=18,
        fontweight="bold",
        y=0.995,
    )
    plt.savefig("comprehensive_dashboard.png", dpi=CONFIG["DPI"], bbox_inches="tight")
    plt.show()
    print("\nComprehensive dashboard saved to: comprehensive_dashboard.png")


print("\n" + "=" * 80)
print("GENERATING VISUALIZATIONS")
print("=" * 80)
print("\n1. Creating equity curve...")
plot_equity_curve(backtest_results, benchmark_subset, CONFIG)
print("\n2. Creating drawdown chart...")
plot_drawdown(backtest_results)
print("\n3. Creating rolling Sharpe ratio...")
plot_rolling_sharpe(backtest_results, window=CONFIG["ROLLING_SHARPE_WINDOW"])
print("\n4. Creating rolling beta...")
plot_rolling_beta(backtest_results, benchmark_subset, window=CONFIG["ROLLING_BETA_WINDOW"])
print("\n5. Creating monthly returns heatmap...")
plot_monthly_returns_heatmap(backtest_results)
print("\n6. Creating annual returns comparison...")
plot_annual_returns_bar(backtest_results, benchmark_subset)
print("\n7. Creating return distribution...")
plot_return_distribution(backtest_results, benchmark_subset)
print("\n8. Creating position analysis...")
plot_position_analysis(backtest_results)
print("\n9. Creating rolling volatility comparison...")
plot_rolling_volatility(backtest_results, benchmark_subset)
print("\n10. Creating comprehensive dashboard...")
create_comprehensive_dashboard(backtest_results, benchmark_subset, metrics, CONFIG)
print(f"\n" + "=" * 80)
print("STRATEGY EXECUTION COMPLETE")
print("=" * 80)
print("\nAll visualizations have been generated and saved:")
print("  1. equity_curve.png")
print("  2. drawdown.png")
print("  3. rolling_sharpe.png")
print("  4. rolling_beta.png")
print("  5. monthly_returns_heatmap.png")
print("  6. annual_returns_bar.png")
print("  7. return_distribution.png")
print("  8. position_analysis.png")
print("  9. rolling_volatility.png")
print("  10. comprehensive_dashboard.png")
print("\nBacktest results saved to: hmm_regime_switching_backtest.csv")
print("=" * 80)
